mnkjkk# Topic 1: Reading Data Sources

> **Phase 5 → Week 8 → Topic 1**
>
> Prerequisites: Phase 3 complete (DataFrames, Spark SQL), Phase 4 complete (Performance)

---

## Phase 5 Overview

Phase 4 taught you HOW Spark runs fast. Phase 5 teaches you HOW to build production ETL pipelines. Real ETL engineers spend most of their time on: reading messy data, enforcing schemas, handling bad records, building incremental loads, writing Delta tables, and making pipelines testable.

---

## The DataFrameReader API

```
spark.read
  .format("csv" | "json" | "parquet" | "orc" | "delta" | "jdbc" | ...)
  .option(key, value)       ← format-specific options
  .schema(schema)           ← explicit schema (or inferSchema=True)
  .load(path)               ← returns DataFrame

# Shortcuts:
spark.read.csv(path, header=True, inferSchema=True)
spark.read.json(path)
spark.read.parquet(path)
```

---

## Interview Cheat Sheet

**Q: Why is `inferSchema=True` dangerous in production?**
> `inferSchema` scans the data to guess types — it's slow (requires an extra pass), non-deterministic (types may change with new data), and wrong for edge cases ("00123" inferred as integer, dropping the leading zeros). In production, always define an explicit `StructType` schema. It's faster, predictable, and fails fast on type mismatches.

**Q: What is `mode` in CSV/JSON reading and what are the options?**
> `mode` controls what Spark does with malformed records: `PERMISSIVE` (default) — puts bad records in a `_corrupt_record` column and fills other fields with null. `DROPMALFORMED` — silently drops bad rows. `FAILFAST` — throws an exception on the first bad row. Production ETL usually uses `PERMISSIVE` + captures `_corrupt_record` to a separate bad-records table.

**Q: What is schema evolution in Parquet/Delta reading?**
> Parquet files written at different times may have different schemas (columns added/removed). `mergeSchema=True` tells Spark to union all schemas found across files — new columns appear with null for files that didn't have them. Without it, Spark uses the schema of the first file and fails or drops columns if later files differ.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json
import os

spark = SparkSession.builder \
    .appName("Week8 - Reading Data Sources") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:10:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


## Part 1: CSV — Options and Bad Record Handling

In [2]:
# Write sample CSV files to /tmp for demo
clean_csv = """\
order_id,customer_id,amount,status,order_date
O001,C001,250.00,delivered,2024-01-15
O002,C002,89.50,pending,2024-01-16
O003,C001,1200.00,delivered,2024-01-17
O004,C003,45.00,cancelled,2024-01-18
"""

messy_csv = """\
order_id,customer_id,amount,status,order_date
O001,C001,250.00,delivered,2024-01-15
O002,C002,NOT_A_NUMBER,pending,2024-01-16
O003,,,delivered,BAD_DATE
O004,C003,45.00,cancelled,2024-01-18
"""

with open("/tmp/clean_orders.csv", "w") as f:
    f.write(clean_csv)
with open("/tmp/messy_orders.csv", "w") as f:
    f.write(messy_csv)

# BAD: inferSchema — scans data, slow, non-deterministic
df_infer = spark.read.csv("/tmp/clean_orders.csv", header=True, inferSchema=True)
print("inferSchema result:")
df_infer.printSchema()

inferSchema result:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)



In [3]:
# GOOD: explicit schema — fast, predictable, fail-fast on mismatch
orders_schema = StructType([
    StructField("order_id",    StringType(),  nullable=False),
    StructField("customer_id", StringType(),  nullable=True),
    StructField("amount",      DoubleType(),  nullable=True),
    StructField("status",      StringType(),  nullable=True),
    StructField("order_date",  DateType(),    nullable=True),
])

df_explicit = spark.read \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(orders_schema) \
    .csv("/tmp/clean_orders.csv")

print("Explicit schema result:")
df_explicit.printSchema()
df_explicit.show()

Explicit schema result:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)



+--------+-----------+------+---------+----------+
|order_id|customer_id|amount|   status|order_date|
+--------+-----------+------+---------+----------+
|    O001|       C001| 250.0|delivered|2024-01-15|
|    O002|       C002|  89.5|  pending|2024-01-16|
|    O003|       C001|1200.0|delivered|2024-01-17|
|    O004|       C003|  45.0|cancelled|2024-01-18|
+--------+-----------+------+---------+----------+



In [4]:
# Reading messy data — the three modes

# Mode 1: PERMISSIVE (default) — bad fields → null, full row in _corrupt_record
df_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(orders_schema.add("_corrupt_record", StringType())) \
    .csv("/tmp/messy_orders.csv")

print("PERMISSIVE mode:")
df_permissive.show(truncate=False)

# Split good vs bad records
good_records = df_permissive.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
bad_records  = df_permissive.filter(F.col("_corrupt_record").isNotNull())

print(f"Good records: {good_records.count()}")
print(f"Bad records:  {bad_records.count()}")

PERMISSIVE mode:


+--------+-----------+------+---------+----------+-----------------------------------------+
|order_id|customer_id|amount|status   |order_date|_corrupt_record                          |
+--------+-----------+------+---------+----------+-----------------------------------------+
|O001    |C001       |250.0 |delivered|2024-01-15|NULL                                     |
|O002    |C002       |NULL  |pending  |2024-01-16|O002,C002,NOT_A_NUMBER,pending,2024-01-16|
|O003    |NULL       |NULL  |delivered|NULL      |O003,,,delivered,BAD_DATE                |
|O004    |C003       |45.0  |cancelled|2024-01-18|NULL                                     |
+--------+-----------+------+---------+----------+-----------------------------------------+



Good records: 4


Bad records:  0


In [5]:
# Mode 2: DROPMALFORMED — silently drops bad rows
df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(orders_schema) \
    .csv("/tmp/messy_orders.csv")

print(f"DROPMALFORMED: {df_drop.count()} rows (bad rows silently dropped)")
df_drop.show()

# Mode 3: FAILFAST — throws exception on first bad row
print("\nFAILFAST mode:")
try:
    df_fail = spark.read \
        .option("header", "true") \
        .option("mode", "FAILFAST") \
        .schema(orders_schema) \
        .csv("/tmp/messy_orders.csv")
    df_fail.show()  # action triggers parsing
except Exception as e:
    print(f"FAILFAST raised: {type(e).__name__}")
    print("(This is expected — bad row detected)")

DROPMALFORMED: 4 rows (bad rows silently dropped)
+--------+-----------+------+---------+----------+---------------+
|order_id|customer_id|amount|   status|order_date|_corrupt_record|
+--------+-----------+------+---------+----------+---------------+
|    O001|       C001| 250.0|delivered|2024-01-15|           NULL|
|    O004|       C003|  45.0|cancelled|2024-01-18|           NULL|
+--------+-----------+------+---------+----------+---------------+


FAILFAST mode:


26/05/16 08:10:23 ERROR Executor: Exception in task 0.0 in stage 14.0 (TID 11)
org.apache.spark.SparkException: [MALFORMED_RECORD_IN_PARSING.WITHOUT_SUGGESTION] Malformed records are detected in record parsing: [O002,C002,null,pending,19738,O002,C002,NOT_A_NUMBER,pending,2024-01-16].
Parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'. 
	at org.apache.spark.sql.errors.QueryExecutionErrors$.malformedRecordsDetectedInRecordParsingError(QueryExecutionErrors.scala:1611)
	at org.apache.spark.sql.catalyst.util.FailureSafeParser.parse(FailureSafeParser.scala:79)
	at org.apache.spark.sql.catalyst.csv.UnivocityParser$.$anonfun$parseIterator$2(UnivocityParser.scala:457)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.ha

FAILFAST raised: Py4JJavaError
(This is expected — bad row detected)


## Part 2: JSON Reading

In [6]:
# Write sample JSON (one JSON object per line = JSON Lines format)
json_lines = [
    {"event_id": "E001", "user_id": "U1", "event": "purchase", "metadata": {"amount": 99.9, "currency": "USD"}, "tags": ["new", "promo"]},
    {"event_id": "E002", "user_id": "U2", "event": "view",     "metadata": {"page": "/home"},                  "tags": []},
    {"event_id": "E003", "user_id": "U1", "event": "click",    "metadata": {"element": "button"},               "tags": ["A/B"]},
]

with open("/tmp/events.json", "w") as f:
    for record in json_lines:
        f.write(json.dumps(record) + "\n")

# Read JSON — Spark auto-detects nested structure
df_json = spark.read.json("/tmp/events.json")
print("JSON schema (auto-detected nested structure):")
df_json.printSchema()
df_json.show(truncate=False)

JSON schema (auto-detected nested structure):
root
 |-- event: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- element: string (nullable = true)
 |    |-- page: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- user_id: string (nullable = true)



+--------+--------+--------------------------+------------+-------+
|event   |event_id|metadata                  |tags        |user_id|
+--------+--------+--------------------------+------------+-------+
|purchase|E001    |{99.9, USD, NULL, NULL}   |[new, promo]|U1     |
|view    |E002    |{NULL, NULL, NULL, /home} |[]          |U2     |
|click   |E003    |{NULL, NULL, button, NULL}|[A/B]       |U1     |
+--------+--------+--------------------------+------------+-------+



In [7]:
# Accessing nested fields
df_json \
    .select(
        "event_id",
        "event",
        F.col("metadata.amount").alias("amount"),
        F.col("metadata.currency").alias("currency"),
        F.col("tags")[0].alias("first_tag")
    ) \
    .show()

# multiLine JSON (entire file is ONE JSON object or array)
multiline_json = '{"orders": [{"id": "O1", "amount": 100}, {"id": "O2", "amount": 200}]}'
with open("/tmp/multiline.json", "w") as f:
    f.write(multiline_json)

df_ml = spark.read \
    .option("multiLine", "true") \
    .json("/tmp/multiline.json")

print("MultiLine JSON:")
df_ml.printSchema()

+--------+--------+------+--------+---------+
|event_id|   event|amount|currency|first_tag|
+--------+--------+------+--------+---------+
|    E001|purchase|  99.9|     USD|      new|
|    E002|    view|  NULL|    NULL|     NULL|
|    E003|   click|  NULL|    NULL|      A/B|
+--------+--------+------+--------+---------+



MultiLine JSON:
root
 |-- orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: long (nullable = true)
 |    |    |-- id: string (nullable = true)



## Part 3: Parquet Reading

In [8]:
# Write Parquet for demo
sample_df = spark.createDataFrame([
    ("O001", "C001", 250.00, "delivered"),
    ("O002", "C002",  89.50, "pending"),
    ("O003", "C001", 1200.00, "delivered"),
], ["order_id", "customer_id", "amount", "status"])

sample_df.write.mode("overwrite").parquet("/tmp/orders_parquet")

# Read Parquet — schema is embedded in the file (no inferSchema needed)
df_parquet = spark.read.parquet("/tmp/orders_parquet")
print("Parquet schema (from file metadata — no inferSchema needed):")
df_parquet.printSchema()
df_parquet.show()

print("""
Why Parquet > CSV for production:
  ✓ Schema embedded in file metadata (no parsing needed)
  ✓ Columnar format — only read columns you need
  ✓ Built-in compression (Snappy by default)
  ✓ Predicate pushdown — Spark skips row groups that don't match filters
  ✓ Splittable — multiple tasks can read one file in parallel
""")

Parquet schema (from file metadata — no inferSchema needed):
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)



+--------+-----------+------+---------+
|order_id|customer_id|amount|   status|
+--------+-----------+------+---------+
|    O003|       C001|1200.0|delivered|
|    O001|       C001| 250.0|delivered|
|    O002|       C002|  89.5|  pending|
+--------+-----------+------+---------+


Why Parquet > CSV for production:
  ✓ Schema embedded in file metadata (no parsing needed)
  ✓ Columnar format — only read columns you need
  ✓ Built-in compression (Snappy by default)
  ✓ Predicate pushdown — Spark skips row groups that don't match filters
  ✓ Splittable — multiple tasks can read one file in parallel



In [9]:
# Schema evolution with mergeSchema
# Write two batches with different schemas
batch1 = spark.createDataFrame([("O001", 100.0)], ["order_id", "amount"])
batch2 = spark.createDataFrame([("O002", 200.0, "delivered")], ["order_id", "amount", "status"])

batch1.write.mode("overwrite").parquet("/tmp/evolving_schema/batch=1")
batch2.write.mode("overwrite").parquet("/tmp/evolving_schema/batch=2")

# Without mergeSchema — uses first file's schema (missing 'status' column)
df_no_merge = spark.read.parquet("/tmp/evolving_schema")
print("Without mergeSchema:")
df_no_merge.printSchema()

# With mergeSchema — union of all schemas
df_merged = spark.read.option("mergeSchema", "true").parquet("/tmp/evolving_schema")
print("With mergeSchema=true:")
df_merged.printSchema()
df_merged.show()

Without mergeSchema:
root
 |-- order_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- batch: integer (nullable = true)

With mergeSchema=true:
root
 |-- order_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- batch: integer (nullable = true)



+--------+------+---------+-----+
|order_id|amount|   status|batch|
+--------+------+---------+-----+
|    O002| 200.0|delivered|    2|
|    O001| 100.0|     NULL|    1|
+--------+------+---------+-----+



## Part 4: Partition Discovery (Hive-style)

In [10]:
# Hive-style partitioned directories: /data/year=2024/month=01/
# Spark auto-discovers these as columns!

for yr, mo, data in [("2024", "01", [("O1", 100.0)]), 
                     ("2024", "02", [("O2", 200.0)]),
                     ("2024", "03", [("O3", 300.0)])]:
    df = spark.createDataFrame(data, ["order_id", "amount"])
    df.write.mode("overwrite").parquet(f"/tmp/partitioned/year={yr}/month={mo}")

# Read the entire partitioned dataset — partition columns appear automatically
df_partitioned = spark.read.parquet("/tmp/partitioned")
print("Partition columns auto-discovered:")
df_partitioned.printSchema()
df_partitioned.show()

# Partition pruning — only reads month=02 directory
df_feb = df_partitioned.filter(F.col("month") == "02")
print("Filtered to month=02 (partition pruning — only reads that directory):")
df_feb.show()

print("""
Partition pruning = Spark skips directories that don't match the filter.
For a 12-month dataset filtered to 1 month = 11/12 less I/O.
""")

Partition columns auto-discovered:
root
 |-- order_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+--------+------+----+-----+
|order_id|amount|year|month|
+--------+------+----+-----+
|      O1| 100.0|2024|    1|
|      O3| 300.0|2024|    3|
|      O2| 200.0|2024|    2|
+--------+------+----+-----+



Filtered to month=02 (partition pruning — only reads that directory):
+--------+------+----+-----+
|order_id|amount|year|month|
+--------+------+----+-----+
|      O2| 200.0|2024|    2|
+--------+------+----+-----+


Partition pruning = Spark skips directories that don't match the filter.
For a 12-month dataset filtered to 1 month = 11/12 less I/O.



## Part 5: JDBC — Reading from Databases

In [11]:
print("""
JDBC Reading — Pattern and Partitioning:
════════════════════════════════════════════════════════════════
# Basic read (single partition — whole table in one task!)
df = spark.read \\
    .format("jdbc") \\
    .option("url", "jdbc:postgresql://host:5432/mydb") \\
    .option("dbtable", "orders") \\
    .option("user", "user") \\
    .option("password", "pass") \\
    .load()

# Parallel read — partition the table by a numeric/date column
df = spark.read \\
    .format("jdbc") \\
    .option("url", "jdbc:postgresql://host:5432/mydb") \\
    .option("dbtable", "orders") \\
    .option("partitionColumn", "order_id")   # must be numeric
    .option("lowerBound", "1")               # range start
    .option("upperBound", "1000000")         # range end
    .option("numPartitions", "20")           # 20 parallel DB connections
    .load()

# Push down predicates to DB (read only what you need)
df = spark.read \\
    .format("jdbc") \\
    .option("dbtable", "(SELECT * FROM orders WHERE status='delivered') t") \\
    ...

WARNING: numPartitions = number of simultaneous DB connections.
Set too high and you'll overwhelm the DB. Check DB max_connections first.
════════════════════════════════════════════════════════════════
""")


JDBC Reading — Pattern and Partitioning:
════════════════════════════════════════════════════════════════
# Basic read (single partition — whole table in one task!)
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://host:5432/mydb") \
    .option("dbtable", "orders") \
    .option("user", "user") \
    .option("password", "pass") \
    .load()

# Parallel read — partition the table by a numeric/date column
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://host:5432/mydb") \
    .option("dbtable", "orders") \
    .option("partitionColumn", "order_id")   # must be numeric
    .option("lowerBound", "1")               # range start
    .option("upperBound", "1000000")         # range end
    .option("numPartitions", "20")           # 20 parallel DB connections
    .load()

# Push down predicates to DB (read only what you need)
df = spark.read \
    .format("jdbc") \
    .option("dbtable", "(SELECT * FROM orders WHERE status='delivered

## Part 6: Key Reading Options Reference

In [12]:
print("""
CSV Reading Options (most important):
  header          true/false       First row is header
  sep             ,                Delimiter (comma, tab, pipe...)
  quote           "                Quote character
  escape          \\               Escape character
  nullValue       ""               String to treat as null
  nanValue        NaN              String to treat as NaN
  dateFormat      yyyy-MM-dd       Date parsing pattern
  timestampFormat yyyy-MM-dd...    Timestamp parsing pattern
  mode            PERMISSIVE       Bad record handling
  encoding        UTF-8            File encoding
  multiLine       false            Allows newlines within quoted fields
  ignoreLeadingWhiteSpace  false   Trim leading spaces from values

JSON Reading Options:
  multiLine       false            File is one JSON object (not JSONL)
  allowComments   false            Allow // comments in JSON
  allowUnquotedFieldNames false    Allow unquoted keys
  mode            PERMISSIVE       Bad record handling
  primitivesAsString false         Read all primitives as strings

Parquet Reading Options:
  mergeSchema     false            Union schemas across files
  recursiveFileLookup false        Read subdirectories recursively
  pathGlobFilter  *.parquet        Only read matching files
  modifiedBefore  2024-01-01       Only read files modified before date
  modifiedAfter   2024-01-01       Only read files modified after date
""")


CSV Reading Options (most important):
  header          true/false       First row is header
  sep             ,                Delimiter (comma, tab, pipe...)
  quote           "                Quote character
  escape          \               Escape character
  nullValue       ""               String to treat as null
  nanValue        NaN              String to treat as NaN
  dateFormat      yyyy-MM-dd       Date parsing pattern
  timestampFormat yyyy-MM-dd...    Timestamp parsing pattern
  mode            PERMISSIVE       Bad record handling
  encoding        UTF-8            File encoding
  multiLine       false            Allows newlines within quoted fields
  ignoreLeadingWhiteSpace  false   Trim leading spaces from values

JSON Reading Options:
  multiLine       false            File is one JSON object (not JSONL)
  allowComments   false            Allow // comments in JSON
  allowUnquotedFieldNames false    Allow unquoted keys
  mode            PERMISSIVE       Bad record hand

## Exercises

1. Read `/workspace/week4/data/orders.csv` using an explicit `StructType` schema (no `inferSchema`). Verify the schema matches the actual data.
2. Write a CSV file with 3 bad rows (wrong type for a numeric column). Read it with `PERMISSIVE` mode, capture `_corrupt_record`, and count good vs bad rows.
3. Write a DataFrame partitioned by `status` to Parquet. Read it back and verify that filtering on `status` uses partition pruning (check `explain()` for `PushedFilters`).
4. What happens when you read a Parquet file and the DataFrame's schema doesn't match? Try adding an extra column to your explicit schema that doesn't exist in the file.
5. Why should you avoid using `inferSchema=True` and `SELECT *` together in the same pipeline? What's the double-danger?

In [13]:
# Exercise 1: Explicit schema read
from pyspark.sql.types import *

orders_schema = StructType([
    StructField("order_id",    StringType(),  True),
    StructField("customer_id", StringType(),  True),
    StructField("product_id",  StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("order_date",  DateType(),    True),
    StructField("status",      StringType(),  True),
    StructField("amount",      DoubleType(),  True),
])

orders = spark.read \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(orders_schema) \
    .csv("/workspace/week4/data/orders.csv")

print("Orders with explicit schema:")
orders.printSchema()
orders.show(5)

Orders with explicit schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)



+--------+-----------+----------+--------+----------+---------+-------+
|order_id|customer_id|product_id|quantity|order_date|   status| amount|
+--------+-----------+----------+--------+----------+---------+-------+
|    O001|          1|      P001|       1|2023-06-01|delivered|1299.99|
|    O002|          1|      P005|       2|2023-06-05|delivered|  79.98|
|    O003|          2|      P002|       3|2023-06-10|delivered|  89.97|
|    O004|          3|      P001|       1|2023-06-12|delivered|1299.99|
|    O005|          3|      P004|       2|2023-06-12|delivered| 599.98|
+--------+-----------+----------+--------+----------+---------+-------+
only showing top 5 rows



In [14]:
# Exercise 3: Partition pruning verification
orders.write.mode("overwrite").partitionBy("status").parquet("/tmp/orders_by_status")

df_by_status = spark.read.parquet("/tmp/orders_by_status")
print("All statuses (partition directories):")
df_by_status.select("status").distinct().show()

# Filter — check explain for PartitionFilters
delivered = df_by_status.filter(F.col("status") == "delivered")
print("Plan for filter on partition column:")
delivered.explain()
print("Look for 'PartitionFilters: [isnotnull(status), (status = delivered)]'")

All statuses (partition directories):


+----------+
|    status|
+----------+
| delivered|
|   shipped|
|processing|
| cancelled|
+----------+

Plan for filter on partition column:
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#529,customer_id#530,product_id#531,quantity#532,order_date#533,amount#534,status#535] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/orders_by_status], PartitionFilters: [isnotnull(status#535), (status#535 = delivered)], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,product_id:string,quantity:int,order_date:date,amount:d...


Look for 'PartitionFilters: [isnotnull(status), (status = delivered)]'
